In [5]:
from IPython.display import Audio, display
import torchaudio

In [121]:
def get_bu_radio_sentence(txt_path):
    with open(txt_path, "r") as f:
        full_str = f.read()
    full_str = " ".join(full_str.split())
    sentences = full_str.split(". ")

    return sentences

def get_bu_radio_words(wrd_path):
    with open(wrd_path, "r") as f:
        lines = [line.strip() for line in f.readlines() if len(line.strip().split()) == 3]
        words = [line.split()[2] for line in lines]

    return words

def get_bu_radio_prosody_tokens(utt_id, tokens_path, tsv_path, st, et, token_rate):
    with open(tsv_path, "r") as f:
        utt_ids = [line.strip().split()[0].split("/")[-1].split(".")[0] for line in f.readlines()[1:]]
        if utt_id in utt_ids:
            index = utt_ids.index(utt_id)
            # Process the tokens for the specific utterance
        else:
            print(f"Utterance ID {utt_id} not found in TSV file.")
    with open(tokens_path, "r") as f:
        tokens = [line.strip() for line in f.readlines()][index]
    tokens = [int(t) for t in tokens.split()]
    # Convert st and et to token indices
    st_token = int(st * token_rate)
    et_token = int(et * token_rate)
    # Extract the tokens for the specified time range
    prosody_tokens = tokens[st_token:et_token]
    return prosody_tokens

In [122]:
utt_id = "f1ajrlp1"
spkr = utt_id[:3]

sph_path= f"/gscratch/tial/data/bu_radio/data/{spkr}/labnews/j/radio/{utt_id}.sph"
speech, sr = torchaudio.load(sph_path)
# play audio
display(Audio(speech.numpy(), rate=sr))


text_path = sph_path.replace(".sph", ".txt")
sentences = get_bu_radio_sentence(text_path)
print(sentences)

wrd_path = sph_path.replace(".sph", ".wrd")
words = get_bu_radio_words(wrd_path)
for i, word in enumerate(words):
    print(f"{i}: {word}")

['Wanted: Chief Justice of the Massachusetts Supreme Court', "brth In April, the S.J.C.'s current leader Edward Hennessy reaches a mandatory retirement age of seventy, brth and a successor is expected to be named in March", 'brth It may be the most important appointment Governor Michael Dukakis makes during the remainder of his administration brth and one of the toughest', "As WBUR's Margo Melnicove reports, brth Hennessy will be a hard act to follow."]
0: Wanted
1: chief
2: Justice
3: of
4: the
5: Massachusetts
6: Supreme
7: court
8: In
9: April
10: the
11: S.J.C.'s
12: current
13: leader
14: Edward
15: Hennessy
16: reaches
17: A
18: mandatory
19: retirement
20: age
21: of
22: seventy
23: and
24: a
25: successor
26: is
27: {expected
28: }to
29: be
30: named
31: In
32: March
33: It
34: may
35: be
36: the
37: most
38: important
39: appointment
40: Governor
41: Michael
42: Dukakis
43: makes
44: during
45: the
46: remainder
47: of
48: his
49: administration
50: and
51: one
52: of
53: the


In [123]:
cutoffs = {"f1ajrlp1": 4, "f2bjrlp1": 4, "m3bjrlp1": 3.2}
prompt_starts_ends = {
    "f1ajrlp1": (4.1, 10),
    "f2bjrlp1": (4.1, 10),
    "m3bjrlp1": (3.3, 8.75)
}
text_cutoffs = {
    "f1ajrlp1": 8,
    "f2bjrlp1": 8,
    "m3bjrlp1": 8,
}
text_prompts_starts_ends = {
    "f1ajrlp1": (8, 23),
    "f2bjrlp1": (8, 23),
    "m3bjrlp1": (8, 23),
}
utt_id2split = {
    "f1ajrlp1": "train",
    "f2bjrlp1": "train",
    "m3bjrlp1": "dev",
}

In [124]:
# trim at cutoff and save
cutoff = cutoffs[utt_id]
trimmed_speech = speech[:, :int(cutoff*sr)]
torchaudio.save(f"/gscratch/tial/kpever/workspace/CosyVoice/bu_radio_example_outputs/inputs/{utt_id}_trimmed.wav", trimmed_speech, sr)

# save text
text_cutoff = text_cutoffs[utt_id]
text_words = words[:text_cutoff]
with open(f"/gscratch/tial/kpever/workspace/CosyVoice/bu_radio_example_outputs/inputs/{utt_id}_trimmed.txt", "w") as f:
    f.write(" ".join(text_words))

In [125]:
# get prosody tokens for speech
split = utt_id2split[utt_id]
tsv_path = f"/gscratch/tial/kpever/workspace/prosodyvec/bu_radio/metadata/{split}.tsv"
tokens_path = f"/gscratch/tial/kpever/workspace/prosodyvec/bu_radio/label/prosodyvec_feats/gigaspeech_glottal_lpf1000_normalized_rawprosody_spectraltilt_targets_spkr_adv_spanloss_wt1en1_lr_5em5_maskprob_0p5_masklen_8/meanpooled_5/{split}.km"
speech_tokens = get_bu_radio_prosody_tokens(
    utt_id,
    tokens_path,
    tsv_path,
    0,
    cutoff,
    token_rate=12.5
)
with open(f"/gscratch/tial/kpever/workspace/CosyVoice/bu_radio_example_outputs/inputs/{utt_id}_trimmed_speech_tokens.txt", "w") as f:
    f.write(" ".join([str(t) for t in speech_tokens]))

In [ ]:
# get prompt
prompt_start = prompt_starts_ends[utt_id][0]
prompt_end = prompt_starts_ends[utt_id][1]
prompt_speech = speech[:, int(prompt_start*sr):int(prompt_end*sr)]
torchaudio.save(f"/gscratch/tial/kpever/workspace/CosyVoice/bu_radio_example_outputs/inputs/{utt_id}_prompt.wav", prompt_speech, sr)

# save prompt text
text_prompt_start = text_prompts_starts_ends[utt_id][0]
text_prompt_end = text_prompts_starts_ends[utt_id][1]
text_prompt_words = words[text_prompt_start:text_prompt_end]
with open(f"/gscratch/tial/kpever/workspace/CosyVoice/bu_radio_example_outputs/inputs/{utt_id}_prompt.txt", "w") as f:
    f.write(" ".join(text_prompt_words))

In [127]:
# get prosody tokens for prompt
split = utt_id2split[utt_id]
tsv_path = f"/gscratch/tial/kpever/workspace/prosodyvec/bu_radio/metadata/{split}.tsv"
tokens_path = f"/gscratch/tial/kpever/workspace/prosodyvec/bu_radio/label/prosodyvec_feats/gigaspeech_glottal_lpf1000_normalized_rawprosody_spectraltilt_targets_spkr_adv_spanloss_wt1en1_lr_5em5_maskprob_0p5_masklen_8/meanpooled_5/{split}.km"
prompt_tokens = get_bu_radio_prosody_tokens(
    utt_id,
    tokens_path,
    tsv_path,
    prompt_start,
    prompt_end,
    token_rate=12.5
)
with open(f"/gscratch/tial/kpever/workspace/CosyVoice/bu_radio_example_outputs/inputs/{utt_id}_prompt_tokens.txt", "w") as f:
    f.write(" ".join([str(t) for t in prompt_tokens]))